# Lesson 01: Hugging Face Datasets + Jev

This is the hands-on orientation to the two main building blocks: loading a labeled dataset from the Hugging Face Hub and asking Jev to score a small sample. You will inspect the dataset first, normalize its labels, take a deterministic pilot sample, then optionally call Jev and review aggregate metrics.

**Setup:** From the repository root run `uv sync`, create `.env` from `.env.example`, and set `TYPESAFE_API_KEY` only for live evaluation. Start with `uv run --env-file .env jupyter lab`. The HF dataset download and the Jev API call are separate network operations.

The dataset is pinned to a Hub revision. The Hub page shows `text` and `label` fields and 662 rows; its Apache-2.0 tag conflicts with embedded CC-BY-4.0 metadata. Do not redistribute prompt rows. Check current source terms before publishing derived results. An 8-row live sample is a workflow exercise, not a statistically meaningful benchmark.

In [ ]:
import os

from jev_eval.datasets import (
    DEEPSET_DATASET_ID,
    DEEPSET_REVISION,
    load_huggingface_split,
)
from jev_eval.metrics import binary_metrics, latency_metrics
from jev_eval.runner import run_evaluation, select_sample

DATASET_ID = DEEPSET_DATASET_ID
REVISION = DEEPSET_REVISION
SPLIT = "test"
PILOT_SIZE = 8
SEED = 42
MODEL = os.getenv("JEV_MODEL", "jev-latest")
RUN_LIVE = False  # Enable only after reviewing expected API usage.
print({"dataset": DATASET_ID, "revision": REVISION, "split": SPLIT, "model": MODEL, "api_key_configured": bool(os.getenv("TYPESAFE_API_KEY"))})

## 1. Load and inspect the dataset

Hugging Face `datasets` loads the requested split from the pinned revision and caches files locally. The package normalizes rows into `Example` objects and rejects unknown label values. The test split is kept separate from future calibration data; do not tune thresholds against this test split.

In [ ]:
examples = load_huggingface_split(
    DATASET_ID, revision=REVISION, split=SPLIT,
)
pilot = select_sample(examples, limit=PILOT_SIZE, seed=SEED)
print({"loaded_rows": len(examples), "pilot_rows": len(pilot)})
print("example id:", pilot[0].example_id)
print("label is injection:", pilot[0].is_injection)
print("text length:", len(pilot[0].text))
# Avoid displaying prompt bodies in shared notebook outputs.

## 2. Check labels and sample deterministically

Before paying for inference, inspect class counts, split names, and a few records locally in a private environment if needed. This cell prints counts and stable IDs, not text. The same seed and source revision should select the same sample order.

In [ ]:
from collections import Counter

print("full split class counts:", Counter(example.is_injection for example in examples))
print("pilot IDs:", [example.example_id for example in pilot])
assert pilot == select_sample(examples, limit=PILOT_SIZE, seed=SEED)

## 3. Optionally evaluate the pilot with Jev

The live runner sends each selected text to Jev. It stores source IDs, hashes, labels, scores, model/version information, token counts, and provider-call timings, but not prompt bodies. The JSONL cache resumes completed rows. Errors are not silently treated as safe.

**Cost and latency:** eight examples mean up to eight model requests. Review the current TypeSafe pricing and your account limits first. This small pilot can show measured latency but cannot establish a p95 service-level objective.

In [ ]:
if RUN_LIVE:
    if not os.getenv("TYPESAFE_API_KEY"):
        raise RuntimeError("Configure TYPESAFE_API_KEY in .env before live evaluation.")
    summary = run_evaluation(
        pilot,
        dataset_id=DATASET_ID,
        revision=REVISION,
        split=SPLIT,
        model=MODEL,
        output_dir=__import__("pathlib").Path("results/runs/lesson-01-pilot"),
        threshold=0.5,
        timeout_seconds=10.0,
    )
    summary
else:
    print("Live Jev evaluation skipped. Set RUN_LIVE=True to make up to 8 API calls.")

In [ ]:
from pathlib import Path
import json

summary_path = Path("results/runs/lesson-01-pilot/summary.json")
if summary_path.exists():
    saved_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "metrics": saved_summary["metrics"],
        "provider_latency": saved_summary["provider_latency"],
        "resolved_models": saved_summary["resolved_models"],
        "provider_p95_under_200ms": saved_summary["provider_p95_under_target"],
    }, indent=2))
else:
    print("No live result found; the next cells use clearly synthetic values.")

## 4. Learn the report shape without a key

When no live summary exists, use this synthetic example to learn the metric schema. These values are fabricated and must never be presented as Jev results. Accuracy on a tiny sample is not a quality claim.

In [ ]:
synthetic_labels = [True, False, True, False]
synthetic_scores = [0.91, 0.12, 0.48, 0.63]
print("SYNTHETIC EXAMPLE ONLY")
print(binary_metrics(synthetic_labels, synthetic_scores, threshold=0.5))
print(latency_metrics([115.0, 144.0, 189.0, 247.0]))

## Takeaways

- A Hugging Face dataset is a versioned source with a schema, splits, labels, and licensing terms; record all of them.
- A deterministic sample helps reproduce a pilot, but production thresholds need a separate calibration set and held-out evaluation.
- Do not publish prompt bodies, credentials, or tiny-sample performance as a benchmark claim.
- Next: [Lesson 02: Thresholds and uncertainty](02_thresholds.ipynb).